# MemeReason dataset tour

A quick look at the extended datasets released with the paper
([QCRI/MemeReason](https://huggingface.co/datasets/QCRI/MemeReason)):
the Hateful Memes and ArMeme benchmarks augmented with natural-language
explanations, fine-grained labels, and distilled chain-of-thought traces.

> **Warning:** the datasets contain memes that may be disturbing or offensive.

In [ ]:
from collections import Counter

from datasets import load_dataset

hateful = load_dataset("QCRI/MemeReason", "hateful_memes")
armeme = load_dataset("QCRI/MemeReason", "armeme")
hateful, armeme

## Label distribution

In [ ]:
for name, ds in [("hateful_memes", hateful), ("armeme", armeme)]:
    print(name)
    for split in ds:
        counts = Counter(ds[split]["label"])
        print(f"  {split:<6} {dict(sorted(counts.items()))}")

## A training example

Training splits carry the full extension: explanation, fine-grained labels, and the distilled `<think>` trace used for CoT supervision.

In [ ]:
example = hateful["train"][11]
print("text:", example["text"])
print("label:", example["label"])
print("protected_category:", example["protected_category"])
print("attack_type:", example["attack_type"])
print("\nexplanation:", example["explanation"])
print("\nthink:", (example["think"] or "")[:600], "...")

In [ ]:
import json

example = armeme["train"][14]
print("text:", example["text"])
print("label:", example["label"])
print("\nexplanation (en):", example["explanation"])
print("\ntechniques:", json.dumps(json.loads(example["techniques"] or "{}"), ensure_ascii=False, indent=2)[:400])

## Length of the distilled reasoning traces

The thinking-length reward $R_{think}$ penalizes traces under 150 words; here is where the distilled supervision actually sits.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=False)
for ax, (name, ds) in zip(axes, [("Hateful Memes", hateful), ("ArMeme", armeme)]):
    lengths = [len(t.split()) for t in ds["train"]["think"] if t]
    ax.hist(lengths, bins=40)
    ax.axvline(150, color="crimson", linestyle="--", label="$L_{min}$ = 150 words")
    ax.set_title(f"{name} - distilled CoT length")
    ax.set_xlabel("words in <think>")
    ax.legend()
plt.tight_layout()

## Next steps

Format this data for training with `data_prep/prepare_training_data.py`, then
follow the pipeline in the repository README: SFT warm-up, supervised GRPO,
and self-supervised GRPO.